# Chapter 06 — Resilient Operations, Disaster Recovery & Rolling Updates

Welcome to **Chapter 06** of the **Google Cloud Dataflow Workshop**!

In this chapter, you will master enterprise operational patterns for running continuous 24/7 streaming systems:
1. **In-Place Zero-Downtime Pipeline Updates (`--update`)**: Upgrading streaming pipelines while preserving in-flight state without data loss.
2. **Graph Compatibility Rules**: The critical role of stable transform names across pipeline versions.
3. **Disaster Recovery with Pub/Sub Seek**: Replaying 24 hours of historical telemetry to backfill corrupted tables without impacting live production.
4. **Poison Pill Defense**: Configuring Pub/Sub Dead-Letter Topics with maximum delivery attempt policies.

## 1. Environment Verification & Kernel Selection

Ensure that your notebook kernel is attached to the project's **`.venv`** managed by **`uv`**:
- Check the top-right kernel selector and select **`Python (dataflow-workshops)`**.

Run the cell below to verify your environment:

In [1]:
import os
import sys

print(f"Python Executable: {sys.executable}")
if ".venv" in sys.executable:
    print("✅ Success: Notebook is running inside the project's .venv!")
else:
    print("⚠️ Warning: Notebook is running on system Python! Select 'Python (dataflow-workshops)' from the top-right kernel picker.")

import apache_beam as beam
print(f"✅ Apache Beam Version: {beam.__version__}")

Python Executable: /Users/zelda.luconi/Medium/google-products/dataflow-workshops/.venv/bin/python3
✅ Success: Notebook is running inside the project's .venv!


✅ Apache Beam Version: 2.77.0


## 2. Architecture & Visual DAG

Here is the architecture for this chapter, illustrating rolling updates and Pub/Sub disaster recovery seek replay:

![Chapter 06 Resilient Operations & Updates DAG](../artifacts/06_resilient_ops_and_updates_dag.png)

<details>
<summary><b>Click to expand Mermaid source code</b></summary>

```mermaid
flowchart TD
    classDef pcoll fill:#e8f0fe,stroke:#4285f4,stroke-width:2px,color:#1a73e8,font-weight:bold;
    classDef ptrans fill:#fef7e0,stroke:#f9ab00,stroke-width:2px,color:#b06000,font-weight:bold;
    classDef sink fill:#e6f4ea,stroke:#34a853,stroke-width:2px,color:#137333,font-weight:bold;
    classDef update fill:#f3e8fd,stroke:#9334e6,stroke-width:2px,color:#6b21a8,font-weight:bold;

    JOB_V1[Running Job V1]:::ptrans --> CHECK{Dataflow Graph Check}:::update
    CHECK --> JOB_V2[Updated Job V2]:::update
    TOPIC[Pub/Sub Topic 7d Retention]:::pcoll --> PROD_SUB[Live Production Sub]:::pcoll
    TOPIC --> RECOV_SUB[Dedicated Recovery Sub]:::update
    PROD_SUB --> JOB_V2 --> PROD_BQ[BigQuery Production]:::sink
    RECOV_SUB --> BACKFILL[Backfill Pipeline]:::update --> TEMP_BQ[BigQuery Temp Staging]:::update
```
</details>

## 3. Inspecting Historical Replay Telemetry

Let's inspect the historical telemetry dataset (`data/replay_telemetry.json`) simulating 24 hours of order transactions:

In [2]:
import json
from pathlib import Path

records = []
with open("data/replay_telemetry.json") as f:
    for i, line in enumerate(f):
        if i < 4:
            records.append(json.loads(line))

print("Sample Historical Replay Records:")
for r in records:
    print(json.dumps(r, indent=2))

Sample Historical Replay Records:
{
  "order_id": "ord-00001",
  "customer_id": "cust_015",
  "subtotal": 291.29,
  "tax_rate": 0.05,
  "order_timestamp": "2026-10-26T19:19:22.076349+00:00"
}
{
  "order_id": "ord-00002",
  "customer_id": "cust_012",
  "subtotal": 117.34,
  "tax_rate": 0.05,
  "order_timestamp": "2026-10-26T19:29:22.076349+00:00"
}
{
  "order_id": "ord-00003",
  "customer_id": "cust_007",
  "subtotal": 174.02,
  "tax_rate": 0.05,
  "order_timestamp": "2026-10-26T19:39:22.076349+00:00"
}
{
  "order_id": "ord-00004",
  "customer_id": "cust_011",
  "subtotal": 292.4,
  "tax_rate": 0.05,
  "order_timestamp": "2026-10-26T19:49:22.076349+00:00"
}


## 4. Concept 1: Graph Compatibility & Transform Name Stability

When performing an in-place update using `--update --job_name=EXISTING_JOB`:
- Dataflow maps running stages to the new pipeline using **Transform Labels** (e.g., `p | 'IngestOrders' >> ...`).
- If transform labels match, Dataflow seamlessly transitions worker threads: in-flight bundles drain on V1 code while new bundles start processing on V2 code.

Notice how both `pipelines/resilient_stream_v1.py` and `pipelines/resilient_stream_v2.py` share exact transform names:
1. `'IngestOrders'`
2. `'ParseOrders'`
3. `'CalculateTaxes'`
4. `'WriteOrdersToSink'`

In [3]:
import inspect
from pipelines import resilient_stream_v1, resilient_stream_v2

print("=== V1 Calculate Taxes Logic ===")
print(inspect.getsource(resilient_stream_v1.CalculateTaxV1Fn))

print("=== V2 Calculate Taxes Logic (with Loyalty Discount) ===")
print(inspect.getsource(resilient_stream_v2.CalculateTaxV2Fn))

=== V1 Calculate Taxes Logic ===
class CalculateTaxV1Fn(beam.DoFn):
    """V1 Tax Processing: standard 5% sales tax."""

    def process(self, order: Dict[str, Any]) -> Iterator[Dict[str, Any]]:
        subtotal = float(order.get("subtotal", 0.0))
        tax = round(subtotal * 0.05, 2)
        total = round(subtotal + tax, 2)
        now_iso = datetime.datetime.now(datetime.timezone.utc).isoformat()

        yield {
            "order_id": str(order.get("order_id")),
            "customer_id": str(order.get("customer_id")),
            "subtotal": subtotal,
            "tax_amount": tax,
            "total_amount": total,
            "pipeline_version": "v1.0.0",
            "processed_at": now_iso,
        }

=== V2 Calculate Taxes Logic (with Loyalty Discount) ===
class CalculateTaxV2Fn(beam.DoFn):
    """V2 Upgraded Tax Processing: includes 10% loyalty discount for orders >= $100."""

    def process(self, order: Dict[str, Any]) -> Iterator[Dict[str, Any]]:
        subtotal = float

## 5. Concept 2: Disaster Recovery with Pub/Sub Seek

When downstream data is corrupted by an application bug:
- **The Wrong Way**: Stopping the production pipeline and seeking the live subscription disruptions all subscribers and causes downstream chaos.
- **The Production Way**:
  1. Ensure the Pub/Sub topic has **Message Retention** enabled (e.g., 7 days).
  2. Create a **new dedicated recovery subscription**.
  3. Use `gcloud pubsub subscriptions seek <RECOVERY_SUB> --time="2026-10-26T14:00:00Z"`.
  4. Backfill into a temporary staging table, validate data, and perform an atomic partition swap!

## 6. Running the Local Rolling Update Simulation

Let's execute `scripts/01_test_rolling_update_simulation.sh` to run both V1 and V2 pipelines locally:

In [4]:
import subprocess

cmd = ["bash", "scripts/01_test_rolling_update_simulation.sh"]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

 Chapter 06: Simulating Rolling Pipeline Update (V1 -> V2)
Input Telemetry: /Users/zelda.luconi/Medium/google-products/dataflow-workshops/06_operations_replay_and_updates/data/replay_telemetry.json
V1 Output:       /Users/zelda.luconi/Medium/google-products/dataflow-workshops/06_operations_replay_and_updates/output/orders_v1*.json
V2 Output:       /Users/zelda.luconi/Medium/google-products/dataflow-workshops/06_operations_replay_and_updates/output/orders_v2*.json
----------------------------------------------------------------------
[1/2] Running Pipeline Version 1 (Standard 5% Tax)...
[2/2] Running Pipeline Version 2 (10% Loyalty Discount on >=$100)...
----------------------------------------------------------------------
 Both versions executed successfully! Generated files in /Users/zelda.luconi/Medium/google-products/dataflow-workshops/06_operations_replay_and_updates/output:
total 96
drwxr-xr-x@ 4 zelda.luconi  staff    128 Oct 27 20:25 .
drwxr-xr-x@ 8 zelda.luconi  staff    256 O

## 7. Inspecting Output Records: V1 vs. V2 Schema Evolution

Notice how V2 seamlessly introduced `loyalty_discount` and `net_amount` while preserving backward compatibility:

In [5]:
import glob
import json

def print_sample(pattern, label):
    print(f"=== {label} ===")
    files = sorted(glob.glob(pattern))
    if files:
        with open(files[0]) as f:
            for line in f.readlines()[:3]:
                print(json.dumps(json.loads(line.strip()), indent=2))
    print()

print_sample("output/orders_v1*.json", "Version 1 Records")
print_sample("output/orders_v2*.json", "Version 2 Records (with Loyalty Discount)")

=== Version 1 Records ===
{
  "order_id": "ord-00001",
  "customer_id": "cust_015",
  "subtotal": 291.29,
  "tax_amount": 14.56,
  "total_amount": 305.85,
  "pipeline_version": "v1.0.0",
  "processed_at": "2026-10-27T19:25:50.412418+00:00"
}
{
  "order_id": "ord-00002",
  "customer_id": "cust_012",
  "subtotal": 117.34,
  "tax_amount": 5.87,
  "total_amount": 123.21,
  "pipeline_version": "v1.0.0",
  "processed_at": "2026-10-27T19:25:50.413738+00:00"
}
{
  "order_id": "ord-00003",
  "customer_id": "cust_007",
  "subtotal": 174.02,
  "tax_amount": 8.7,
  "total_amount": 182.72,
  "pipeline_version": "v1.0.0",
  "processed_at": "2026-10-27T19:25:50.413789+00:00"
}

=== Version 2 Records (with Loyalty Discount) ===
{
  "order_id": "ord-00001",
  "customer_id": "cust_015",
  "subtotal": 291.29,
  "loyalty_discount": 29.13,
  "net_amount": 262.16,
  "tax_amount": 13.11,
  "total_amount": 275.27,
  "pipeline_version": "v2.0.0",
  "processed_at": "2026-10-27T19:25:53.883120+00:00"
}
{
  "orde

## 8. Summary & Next Steps

### What You Learned in Chapter 06:
1. **In-Place Zero-Downtime Updates**: Using `--update` to deploy new code without stopping pipelines or risking duplicate processing.
2. **Graph Compatibility**: Maintaining stable transform labels and schema backward compatibility.
3. **Pub/Sub Subscription Seek**: Backfilling historical data using dedicated recovery subscriptions without interrupting production consumers.
4. **Poison Pill Defense**: Using Pub/Sub Dead-Letter Topics with maximum delivery attempt limits.

### Deploying to Google Cloud Dataflow:
To deploy V1 and perform an in-place update on GCP, run:

```bash
bash scripts/02_deploy_v1_and_update_v2.sh
```

In **Chapter 07**, we will explore **Unit Testing, Docker Flex Templates, CI/CD & Security** (`TestPipeline`, `PAssert`, `TestStream`, Artifact Registry Flex Templates, CMEK, and Private IPs).